In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

pd.set_option('display.width', None)

df = pd.read_parquet('data/processed/distribution_analysis.parquet')

df.head()

,impl_strike,F,f,Ftilda,Date,secid,price,base_price,return,spot_return,impl_vol,impl_vol_der,traded_range,delta,step,event_date,window_position,maturity_days
0,597.517175,0.058505,0.002951,0.053111,1996-01-30,108105,632.68438,632.68438,-0.055584,0.0,0.157460,-0.001529,1,91.771394,0.00001,1996-01-31,t-1,30
1,597.660835,0.058932,0.002975,0.053515,1996-01-30,108105,632.68438,632.68438,-0.055357,0.0,0.157240,-0.001529,1,91.670822,0.00001,1996-01-31,t-1,30
2,597.804496,0.059363,0.003000,0.053922,1996-01-30,108105,632.68438,632.68438,-0.055130,0.0,0.157021,-0.001529,1,91.570242,0.00001,1996-01-31,t-1,30
3,597.948156,0.059798,0.003025,0.054332,1996-01-30,108105,632.68438,632.68438,-0.054903,0.0,0.156801,-0.001529,1,91.469646,0.00001,1996-01-31,t-1,30
4,598.091816,0.060236,0.003050,0.054746,1996-01-30,108105,632.68438,632.68438,-0.054676,0.0,0.156581,-0.001529,1,91.369026,0.00001,1996-01-31,t-1,30


In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

pd.set_option('display.width', None)

df = pd.read_parquet('data/processed/quantile_mapping_analysis.parquet')

df.head()

,impl_strike,F,f,Ftilda,Date,secid,price,base_price,return,spot_return,impl_vol,impl_vol_der,traded_range,delta,step,event_date,window_position,maturity_days,g,mapping_period,sccr
0,598.235477,0.060677,0.003075,0.055164,1996-01-30,108105,632.68438,632.68438,-0.054449,0.009297,0.156362,-0.001529,1,91.268373,0.00001,1996-01-31,t-1,30,602.923891,t-1_to_t0,0.007837
1,598.379137,0.061123,0.003100,0.055585,1996-01-30,108105,632.68438,632.68438,-0.054222,0.009297,0.156142,-0.001529,1,91.167681,0.00001,1996-01-31,t-1,30,603.070441,t-1_to_t0,0.007840
2,598.522798,0.061572,0.003126,0.056010,1996-01-30,108105,632.68438,632.68438,-0.053995,0.009297,0.155922,-0.001529,1,91.066941,0.00001,1996-01-31,t-1,30,603.217088,t-1_to_t0,0.007843
3,598.666458,0.062024,0.003151,0.056438,1996-01-30,108105,632.68438,632.68438,-0.053768,0.009297,0.155702,-0.001529,1,90.966146,0.00001,1996-01-31,t-1,30,603.363833,t-1_to_t0,0.007846
4,598.810118,0.062481,0.003177,0.056870,1996-01-30,108105,632.68438,632.68438,-0.053541,0.009297,0.155483,-0.001529,1,90.865287,0.00001,1996-01-31,t-1,30,603.510676,t-1_to_t0,0.007850


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib import rcParams
import os

os.makedirs('results', exist_ok=True)


def set_academic_style():
    rcParams.update({
        'font.family': 'serif',
        'font.serif': ['Times New Roman', 'DejaVu Serif'],
        'font.size': 11,
        'mathtext.fontset': 'stix',
        'figure.figsize': (6.5, 4.5),
        'figure.dpi': 300,
        'axes.linewidth': 0.8,
        'axes.edgecolor': '#333333',
        'axes.labelcolor': '#333333',
        'axes.titlesize': 12,
        'axes.labelsize': 11,
        'xtick.direction': 'out',
        'ytick.direction': 'out',
        'xtick.color': '#333333',
        'ytick.color': '#333333',
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        'legend.frameon': True,
        'legend.framealpha': 0.9,
        'legend.edgecolor': '#DDDDDD',
        'grid.color': '#EEEEEE',
        'grid.linestyle': '-',
        'grid.linewidth': 0.7,
        'grid.alpha': 0.8,
        'lines.linewidth': 1.5,
        'lines.markersize': 4,
        'savefig.dpi': 300,
        'savefig.bbox': 'tight',
        'savefig.pad_inches': 0.1,
        'savefig.format': 'pdf'
    })

PANTONE_COLORS = {
    'ultra_violet_2018': '#5F4B8B',
    'peach_fuzz_2024': '#FFBE98',
    'soft_teal': '#7A9E9F',
    'dusty_lavender': '#B8A9C9',
    'warm_sand': '#D7CEC7',
    'pale_moss': '#C8D5B9',
    'misty_blue': '#96B3D0'
}

set_academic_style()

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import PercentFormatter
import os
from scipy import interpolate
from scipy.ndimage import gaussian_filter1d

os.makedirs('results/figure1', exist_ok=True)

df_dist = pd.read_parquet('data/processed/distribution_analysis.parquet')
df_panel = pd.read_parquet('data/processed/options_fomc_panel_1996_2023.parquet')

df_dist['Date'] = pd.to_datetime(df_dist['Date'])
df_panel['Date'] = pd.to_datetime(df_panel['Date'])

# Get all trading dates (for locating previous and next trading days)
full_dates = pd.to_datetime(df_panel['Date'].unique())
full_dates = np.sort(full_dates)

maturities = [30, 91, 182, 365]

def get_prev_next_dates(event_date):
    """Get the previous and next trading days of the event date."""
    event_date = pd.to_datetime(event_date)
    # Note: full_dates is a numpy.datetime64 array, need to convert to pandas Timestamp for comparison
    # Use numpy comparison directly, convert event_date to numpy.datetime64
    event_np = np.datetime64(event_date)
    idx = np.where(full_dates == event_np)[0]
    if len(idx) == 0:
        return None, None
    i = idx[0]
    prev_date = full_dates[i-1] if i > 0 else None
    next_date = full_dates[i+1] if i < len(full_dates)-1 else None
    return prev_date, next_date

def plot_single_event(event_date, event_name, save_filename):
    """Plot IV curve comparison (t+1 vs t-1) for a single FOMC event (four subplots)"""
    event_date = pd.to_datetime(event_date)
    prev_date, next_date = get_prev_next_dates(event_date)
    if prev_date is None or next_date is None:
        print(f"Cannot find previous and next trading days for {event_date.strftime('%Y-%m-%d')}, skipping")
        return
    # Convert to pandas Timestamp to use strftime
    prev_date = pd.to_datetime(prev_date)
    next_date = pd.to_datetime(next_date)
    print(f"Plotting {event_name} ({event_date.strftime('%Y-%m-%d')}): t-1={prev_date.strftime('%Y-%m-%d')}, t+1={next_date.strftime('%Y-%m-%d')}")
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.flatten()
    
    df_dist_reduced = df_dist[['Date', 'maturity_days', 'delta', 'impl_vol']]
    delta_grid = np.linspace(10, 90, 50)
    
    for i, maturity in enumerate(maturities):
        ax = axes[i]
        maturity_data = df_dist_reduced[df_dist_reduced['maturity_days'] == maturity]
        
        # Extract previous day and next day data
        prev_day_data = maturity_data[maturity_data['Date'] == prev_date]
        next_day_data = maturity_data[maturity_data['Date'] == next_date]
        
        if len(prev_day_data) < 5 or len(next_day_data) < 5:
            ax.text(0.5, 0.5, f"Insufficient data for {maturity} days",
                    ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{maturity}-Day Options')
            continue
        
        prev_day_data = prev_day_data.sort_values('delta')
        next_day_data = next_day_data.sort_values('delta')
        
        try:
            prev_interp = interpolate.interp1d(
                prev_day_data['delta'], prev_day_data['impl_vol'],
                kind='linear', bounds_error=False, fill_value='extrapolate'
            )
            next_interp = interpolate.interp1d(
                next_day_data['delta'], next_day_data['impl_vol'],
                kind='linear', bounds_error=False, fill_value='extrapolate'
            )
            prev_vols = prev_interp(delta_grid) * 100
            next_vols = next_interp(delta_grid) * 100
            
            sigma = 1.5
            smooth_prev = gaussian_filter1d(prev_vols, sigma)
            smooth_next = gaussian_filter1d(next_vols, sigma)
            
            ax.plot(delta_grid, smooth_prev, color=PANTONE_COLORS['peach_fuzz_2024'],
                    label=f't-1 ({prev_date.strftime("%Y-%m-%d")})', linewidth=2.5)
            ax.plot(delta_grid, smooth_next, color=PANTONE_COLORS['ultra_violet_2018'],
                    label=f't+1 ({next_date.strftime("%Y-%m-%d")})', linewidth=2.5)
            
            ax.set_xlabel('Delta', fontsize=13, fontweight='bold')
            ax.set_ylabel('Implied Volatility (%)', fontsize=15, fontweight='bold')
            ax.set_title(f'{maturity}-Day Options', fontsize=16, fontweight='bold')
            ax.grid(True, alpha=0.3, linestyle='--')
            ax.yaxis.set_major_formatter(PercentFormatter(decimals=0))
            ax.set_xlim(10, 90)
            ax.tick_params(axis='both', which='major', labelsize=14)
            
            if i == 0:
                ax.legend(frameon=True, fancybox=True, shadow=True, framealpha=0.9,
                          loc='best', fontsize=15, handlelength=2, handletextpad=0.5)
        except Exception as e:
            ax.text(0.5, 0.5, f"Interpolation error: {e}", ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{maturity}-Day Options')
    
    plt.tight_layout()
    plt.savefig(save_filename, bbox_inches='tight', dpi=300)
    plt.close()
    print(f"Saved: {save_filename}")

# Define two typical events
events = [
    ('2008-10-29', '2008 Fed Put (Oct 29, 2008)'),
    ('2020-03-23', '2020 Fed Put (Mar 23, 2020)')
]

for date_str, name in events:
    event_date = pd.to_datetime(date_str)
    filename = f'results/{date_str}_fedput_iv_smile.png'
    plot_single_event(event_date, name, filename)

print("All visualizations completed!")

Plotting 2008 Fed Put (Oct 29, 2008) (2008-10-29): t-1=2008-10-28, t+1=2008-10-30
Saved: results/2008-10-29_fedput_iv_smile.png
Plotting 2020 Fed Put (Mar 23, 2020) (2020-03-23): t-1=2020-03-20, t+1=2020-03-24
Saved: results/2020-03-23_fedput_iv_smile.png
All visualizations completed!


In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

def load_data():
    """Load data"""
    skew_data = pd.read_parquet('data/processed/skewness_analysis.parquet')
    fomc_data = pd.read_parquet('data/processed/options_fomc_panel_1996_2023.parquet')
    return skew_data, fomc_data

def get_special_periods(fomc_data):
    """Identify special periods (high VIX × negative MPS × FOMC days)"""
    return fomc_data[
        (fomc_data['vix_stress_75'] == 1) & 
        (fomc_data['monetary_policy_shock'] < 0) &
        (fomc_data['fomc_dummy'] == 1)
    ]['Date'].unique()

def get_fomc_dates_2008(fomc_data):
    """Get all FOMC dates in 2008 with their special status"""
    fomc_2008 = fomc_data[
        (fomc_data['Date'] >= '2008-01-01') & 
        (fomc_data['Date'] <= '2008-12-31') &
        (fomc_data['fomc_dummy'] == 1)
    ][['Date', 'vix_stress_75', 'monetary_policy_shock']].drop_duplicates()
    
    fomc_2008['is_special'] = (fomc_2008['vix_stress_75'] == 1) & (fomc_2008['monetary_policy_shock'] < 0)
    return fomc_2008

def plot_full_sample_with_spx(skew_data, fomc_data, special_dates, save_dir):
    """Full sample time series with SPX on secondary axis (30-day skew only)"""
    daily_spx = fomc_data.groupby('Date')['spx_price'].first().reset_index()
    
    fig, ax1 = plt.subplots(figsize=(14, 8))
    ax2 = ax1.twinx()
    
    # SPX: subtle background
    ax2.plot(daily_spx['Date'], daily_spx['spx_price'], color='#C8D5B9', linewidth=1.5, 
             alpha=0.6, label='SPX', zorder=1)
    
    # 30-day skewness only
    maturity_data = skew_data[skew_data['days'] == 30].sort_values('Date')
    ax1.plot(maturity_data['Date'], maturity_data['SKEW'], 
             color='#5F4B8B', label='30 days', linewidth=1.5, zorder=2)
    
    # Special FOMC periods
    for date in special_dates:
        ax1.axvline(x=date, color='#8DA0CB', linestyle='--', alpha=0.7, linewidth=0.8, zorder=3)
    
    ax1.set_ylabel('Option-Implied Lower Skewness', fontsize=12)
    ax2.set_ylabel('SPX Price', fontsize=12)
    
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    ax1.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    plt.savefig(os.path.join(save_dir, 'full_sample_with_spx.png'), dpi=300)
    plt.close()

def plot_2008_financial_crisis(skew_data, fomc_data, save_dir):
    """Plot 2008 financial crisis skewness data (30-day and 365-day) with VIX and FOMC events"""
    data_2008 = skew_data[
        (skew_data['Date'] >= '2008-08-01') & 
        (skew_data['Date'] <= '2008-12-31')
    ]
    daily_vix_2008 = fomc_data[
        (fomc_data['Date'] >= '2008-08-01') & 
        (fomc_data['Date'] <= '2008-12-31')
    ].groupby('Date')['vix'].first().reset_index()
    fomc_dates_2008 = get_fomc_dates_2008(fomc_data)
    
    if data_2008.empty:
        print("No data found for 2008")
        return
    
    fig, ax1 = plt.subplots(figsize=(14, 8))
    ax2 = ax1.twinx()
    
    ax2.plot(daily_vix_2008['Date'], daily_vix_2008['vix'], 
             color='#8B0000', linestyle='-.', linewidth=2.0, alpha=0.5, label='VIX', zorder=1)
    
    colors = {30: '#5F4B8B', 365: '#D7CEC7'}
    for days in [30, 365]:
        maturity_data = data_2008[data_2008['days'] == days].sort_values('Date')
        if not maturity_data.empty:
            ax1.plot(maturity_data['Date'], maturity_data['SKEW'], 
                    color=colors[days], label=f'{days} days', linewidth=1.5, zorder=2)
    
    special_count = 0
    regular_count = 0
    for _, row in fomc_dates_2008.iterrows():
        date = row['Date']
        if pd.to_datetime('2008-08-01') <= date <= pd.to_datetime('2008-12-31'):
            if row['is_special']:
                ax1.axvline(x=date, color='#8DA0CB', linestyle='--', alpha=0.8, linewidth=2)
                special_count += 1
            else:
                ax1.axvline(x=date, color='#8DA0CB', linestyle='--', alpha=0.4, linewidth=1)
                regular_count += 1
    print(f"2008 FOMC events (Aug-Dec): {special_count} special, {regular_count} regular")
    
    ax1.set_ylabel('Option Implied Skewness', fontsize=12)
    ax2.set_ylabel('VIX', fontsize=12)
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    ax1.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    plt.savefig(os.path.join(save_dir, '2008_financial_crisis.png'), dpi=300)
    plt.close()

def plot_march_2020_events(skew_data, fomc_data, save_dir):
    """Plot March 2020 skewness data (30-day and 365-day) with VIX and event annotations"""
    data_march_2020 = skew_data[
        (skew_data['Date'] >= '2020-03-01') & 
        (skew_data['Date'] <= '2020-03-31')
    ]
    daily_vix_march_2020 = fomc_data[
        (fomc_data['Date'] >= '2020-03-01') & 
        (fomc_data['Date'] <= '2020-03-31')
    ].groupby('Date')['vix'].first().reset_index()
    
    fomc_dates_march_2020 = fomc_data[
        (fomc_data['Date'] >= '2020-03-01') & 
        (fomc_data['Date'] <= '2020-03-31') &
        (fomc_data['fomc_dummy'] == 1)
    ][['Date', 'vix_stress_75', 'monetary_policy_shock']].drop_duplicates()
    fomc_dates_march_2020['is_special'] = (fomc_dates_march_2020['vix_stress_75'] == 1) & (fomc_dates_march_2020['monetary_policy_shock'] < 0)
    
    if data_march_2020.empty:
        print("No data found for March 2020")
        return
    
    fig, ax1 = plt.subplots(figsize=(14, 8))
    ax2 = ax1.twinx()
    
    ax2.plot(daily_vix_march_2020['Date'], daily_vix_march_2020['vix'], 
             color='#8B0000', linestyle='-.', linewidth=2.0, alpha=0.5, label='VIX', zorder=1)
    
    colors = {30: '#5F4B8B', 365: '#D7CEC7'}
    for days in [30, 365]:
        maturity_data = data_march_2020[data_march_2020['days'] == days].sort_values('Date')
        if not maturity_data.empty:
            ax1.plot(maturity_data['Date'], maturity_data['SKEW'], 
                    color=colors[days], label=f'{days} days', linewidth=2, zorder=2)
    
    ymin, ymax = ax1.get_ylim()
    text_y = ymin + 0.036 * (ymax - ymin)
    
    events = {
        '2020-03-03': ('Unscheduled FOMC Meeting', 0.5),
        '2020-03-15': ('Unscheduled FOMC Meeting', 0.5),
        '2020-03-23': ('Corporate Bond Purchases Announced', 0.8)
    }
    for event_date, (event_name, alpha) in events.items():
        event_date_dt = pd.to_datetime(event_date)
        ax1.axvline(x=event_date_dt, color='#8DA0CB', linestyle='--', linewidth=2, alpha=alpha)
        x_offset = pd.Timedelta(days=0.2)
        ax1.text(event_date_dt + x_offset, text_y, event_name, 
                color='#8DA0CB', alpha=alpha, fontsize=9,
                ha='left', va='top', rotation=0)
    
    ax1.set_ylabel('Option Implied Skewness', fontsize=12)
    ax2.set_ylabel('VIX', fontsize=12)
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
    ax1.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    plt.savefig(os.path.join(save_dir, 'march_2020_events.png'), dpi=300)
    plt.close()

def plot_term_structure_inversion(skew_data, fomc_data, save_dir):
    """
    Plot term structure inversion for 2008 crisis.
    Left: panic day (highest VIX in October 2008).
    Right: normal day (last trading day in December 2008).
    """
    # ----- Panic day: max VIX in October 2008 -----
    vix_oct = fomc_data[
        (fomc_data['Date'] >= '2008-10-01') & 
        (fomc_data['Date'] <= '2008-10-31')
    ].groupby('Date')['vix'].first().reset_index()
    
    if vix_oct.empty:
        print("No VIX data for October 2008, skipping term structure plot.")
        return
    
    panic_idx = vix_oct['vix'].idxmax()
    panic_date = vix_oct.loc[panic_idx, 'Date']
    
    # ----- Normal day: last trading day in December 2008 -----
    vix_dec = fomc_data[
        (fomc_data['Date'] >= '2008-12-01') & 
        (fomc_data['Date'] <= '2008-12-31')
    ].groupby('Date')['vix'].first().reset_index()
    
    if vix_dec.empty:
        print("No VIX data for December 2008, skipping term structure plot.")
        return
    
    normal_date = vix_dec['Date'].max()   # last trading day of the month
    
    # Maturities to plot
    maturity_list = [30, 91, 182, 365]
    x_labels = [f'{m}d' for m in maturity_list]
    x = np.arange(len(maturity_list))
    
    def get_skew_for_date(date, df):
        d = pd.to_datetime(date)
        sub = df[df['Date'] == d]
        skews = []
        for m in maturity_list:
            vals = sub[sub['days'] == m]['SKEW']
            if len(vals) > 0:
                skews.append(vals.values[0])
            else:
                skews.append(np.nan)
        return skews
    
    # Create 1×2 subplots
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    line_color = '#4C72B0'
    fill_alpha = 0.15
    marker_size = 8
    line_width = 2.5
    
    def style_line_plot(ax, skews, title_str):
        if np.isnan(skews).all():
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        else:
            ax.plot(x, skews, color=line_color, marker='o', linewidth=line_width,
                    markersize=marker_size, markerfacecolor='white', markeredgewidth=1.5,
                    markeredgecolor=line_color)
            ax.fill_between(x, skews, alpha=fill_alpha, color=line_color)
            ax.set_xticks(x)
            ax.set_xticklabels(x_labels)
            ax.set_ylabel('Lower Skewness')
            ax.grid(True, alpha=0.3, linestyle='--')
            ax.axhline(y=0, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
        ax.set_title(title_str, fontweight='bold')
    
    # Panic day
    skews_panic = get_skew_for_date(panic_date, skew_data)
    style_line_plot(axes[0], skews_panic, 
                    f'Panic Day: {pd.to_datetime(panic_date).strftime("%Y-%m-%d")} ')
    
    # Normal day
    skews_normal = get_skew_for_date(normal_date, skew_data)
    style_line_plot(axes[1], skews_normal,
                    f'Normal Day: {pd.to_datetime(normal_date).strftime("%Y-%m-%d")}')
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'term_structure_inversion.png'), dpi=300)
    plt.close()
    print(f"Term structure plot saved (panic={panic_date}, normal={normal_date})")
    
def main():
    save_dir = 'results'
    os.makedirs(save_dir, exist_ok=True)
    
    skew_data, fomc_data = load_data()
    special_dates = get_special_periods(fomc_data)
    
    print(f"Found {len(special_dates)} special periods meeting all criteria")
    
    print("Generating full sample time series with SPX (30-day only)...")
    plot_full_sample_with_spx(skew_data, fomc_data, special_dates, save_dir)
    
    print("Generating 2008 financial crisis plot (30d & 365d) with VIX and FOMC events...")
    plot_2008_financial_crisis(skew_data, fomc_data, save_dir)
    
    print("Generating March 2020 events plot (30d & 365d) with VIX...")  
    plot_march_2020_events(skew_data, fomc_data, save_dir)
    
    print("Generating term structure inversion plot (panic vs normal days)...")
    plot_term_structure_inversion(skew_data, fomc_data, save_dir)
    
    print(f"All plots saved to: {save_dir}")

if __name__ == "__main__":
    main()

Found 28 special periods meeting all criteria
Generating full sample time series with SPX (30-day only)...
Generating 2008 financial crisis plot (30d & 365d) with VIX and FOMC events...
2008 FOMC events (Aug-Dec): 2 special, 3 regular
Generating March 2020 events plot (30d & 365d) with VIX...
Generating term structure inversion plot (panic vs normal days)...
Term structure plot saved (panic=2008-10-27 00:00:00, normal=2008-12-31 00:00:00)
All plots saved to: results
